In [15]:
from analyses.single_neuron_analysis import run_multi_predictor_glm
from analyses.spike_count import get_binned_spike_trials, aggregate_trial_level, extract_spike_counts_from_cells
import pandas as pd
import matplotlib
import seaborn as sns
import matplotlib.pyplot as plt
# date = '2023-09-26'
# round_no = 1
# analysis_df = prepare_binned_spike_data(date, round_no, 0.05)
# spike_df = aggregate_trial_level(analysis_df)

sig_neurons = pd.read_pickle(f'/home/connorlab/Documents/GitHub/Julie/Cortana/analysis_cache/Zombies_significant_neurons_pANOVAorGLM_passed.pkl')
neuron_df = extract_spike_counts_from_cells(sig_neurons)
neuron_df

Extracting spike counts: 100%|██████████| 319/319 [00:00<00:00, 522.45it/s]


,NeuronID,MonkeyName,MonkeyGroup,TaskField,SpikeCount
0,AMG_2023-09-26_1_Channel.C_014_Unit 1,114J,Instigators,1695747327712000,146
1,AMG_2023-09-26_1_Channel.C_014_Unit 1,69X,Zombies,1695747327797000,105
2,AMG_2023-09-26_1_Channel.C_014_Unit 1,59E,Instigators,1695747327892000,92
3,AMG_2023-09-26_1_Channel.C_014_Unit 1,40J,Stranger Things,1695747327973000,91
4,AMG_2023-09-26_1_Channel.C_014_Unit 1,40J,Stranger Things,1695747328008000,87
...,...,...,...,...,...
116210,Unknown_2023-12-18_3_Channel.C_029_Unit 1,40J,Stranger Things,1702937483729000,17
116211,Unknown_2023-12-18_3_Channel.C_029_Unit 1,G701,Best Frans,1702937494561000,11
116212,Unknown_2023-12-18_3_Channel.C_029_Unit 1,42Z,Instigators,1702937494973000,8
116213,Unknown_2023-12-18_3_Channel.C_029_Unit 1,35Y,Instigators,1702937495986000,10


In [24]:
formula = """
SpikeCount ~ AffiliationFrom_z + AffiliationTo_z +
              AgonismFrom_z + AgonismTo_z +
              SubmissionFrom_z + SubmissionTo_z
"""

# formula = """
# SpikeCount ~ Rank + C(Sex) + C(Family) + C(Grouping)
# """

In [25]:
behavior_df= pd.read_excel('/home/connorlab/Documents/GitHub/Julie/social_data/zombies_social_data/zombies_marginals_without_cortana.xlsx')

In [26]:
from scipy.stats import zscore

for col in ['AffiliationFrom', 'AffiliationTo', 'AgonismFrom', 'AgonismTo', 'SubmissionFrom', 'SubmissionTo', ]:
    behavior_df[col + "_z"] = zscore(behavior_df[col])

In [27]:
# formula = "SpikeCount ~ PC1 + PC2"

In [28]:
# df = spike_df.merge(behavior_df, on="MonkeyName")
df = neuron_df.merge(behavior_df, on="MonkeyName")

In [29]:
significant, real_sig = run_multi_predictor_glm(df, formula, neuron_col="NeuronID")


Running multi-GLM per neuron: 100%|██████████| 319/319 [00:02<00:00, 131.66it/s]


In [34]:
# er_df = significant[significant['NeuronID'].str.startswith('ER')].copy()
# amg_df = significant[significant['NeuronID'].str.startswith('AMG')].copy()
top_neurons = (
    significant
    .sort_values(by='P>|z|')                # p-value 오름차순
    .groupby('NeuronID')                    # 뉴런 단위로
    .first()                                # 가장 작은 p값만 대표로 사용
    .sort_values(by='P>|z|')                # 다시 정렬
    .head(50)                               # 상위 50개 선택
    .index                                  # NeuronID만 추출
)
# reshape for heatmap
# heatmap_er_df = significant.pivot(index='NeuronID', columns='index', values='Coef.')

# Step 2: 해당 뉴런만 필터링해서 heatmap용 pivot 테이블 만들기
subset = significant[significant['NeuronID'].isin(top_neurons)]
heatmap_subset = subset.pivot(index='NeuronID', columns='index', values='Coef.')

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_subset, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Significant GLM Coefficients per Neuron")
plt.xlabel("Variable")
plt.ylabel("NeuronID")
plt.tight_layout()
plt.show()

In [37]:
heatmap_subset

index,AffiliationFrom_z,AffiliationTo_z,AgonismFrom_z,AgonismTo_z,Intercept,SubmissionFrom_z,SubmissionTo_z
NeuronID,,,,,,,
AMG_2023-09-26_1_Channel.C_014_Unit 1,-0.588625,0.544117,-0.207608,-0.073234,4.610375,0.001568,0.388974
AMG_2023-09-26_1_Channel.C_027_Unit 1,-5.035687,4.360550,0.038281,-1.179072,4.301139,0.727461,1.581616
AMG_2023-09-26_3_Channel.C_027_Unit 2,2.052582,-1.764349,-0.057081,0.524309,3.854155,-0.311068,-0.593667
AMG_2023-09-28_1_Channel.C_025_Unit 1,-3.018075,2.569669,-0.128425,-0.698986,4.302077,0.348959,1.166028
AMG_2023-09-29_3_Channel.C_025,1.287974,-1.118208,-0.089207,0.340335,3.735434,-0.194995,-0.447870
AMG_2023-10-03_1_Channel.C_009,8.767904,-7.532808,-0.327855,2.284223,3.931707,-1.281877,-2.266330
AMG_2023-10-03_1_Channel.C_019,-17.816593,15.357698,0.323189,-4.158001,5.663805,2.196550,4.960511
AMG_2023-10-03_1_Channel.C_028,-39.082435,33.396560,0.967802,-9.548053,6.902627,5.028296,10.490192
AMG_2023-10-03_3_Channel.C_004,-6.943351,5.950034,0.123119,-1.573305,4.119807,0.830135,1.931766


In [36]:
# reshape for heatmap
heatmap_amg_df = amg_df.pivot(index='NeuronID', columns='index', values='Coef.')

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_amg_df, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Significant GLM Coefficients per Neuron in AMG")
plt.xlabel("Variable")
plt.ylabel("NeuronID")
plt.tight_layout()
plt.show()

KeyboardInterrupt: 

In [21]:
amg_sig_count = amg_df['index'].value_counts().sort_index()

plt.figure(figsize=(10, 4))
sns.barplot(x=amg_sig_count.index, y=amg_sig_count.values)
plt.ylabel("Number of Significant Neurons")
plt.title("Which Variables Drive Neural Firing in AMG?")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [22]:
er_sig_count = er_df['index'].value_counts().sort_index()

plt.figure(figsize=(10, 4))
sns.barplot(x=er_sig_count.index, y=er_sig_count.values)
plt.ylabel("Number of Significant Neurons")
plt.title("Which Variables Drive Neural Firing in ER?")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()